# Module 4 — Auto Loader + Structured Streaming
Exam domain: **Data Processing**

Databricks notebook — uses the real `cloudFiles` Auto Loader source.

In [ ]:
dbutils.widgets.text("source_path", "/Volumes/main/module4/landing")
dbutils.widgets.text("checkpoint_path", "/Volumes/main/module4/_checkpoints/events")
dbutils.widgets.text("target_table", "main.module4.bronze_events_stream")
source_path = dbutils.widgets.get("source_path")
checkpoint_path = dbutils.widgets.get("checkpoint_path")
target_table = dbutils.widgets.get("target_table")

## Auto Loader with schema inference + evolution
`cloudFiles.schemaLocation` stores the inferred schema so it persists across
restarts. `schemaEvolutionMode = "addNewColumns"` means new columns fail the
current micro-batch once, then are added automatically on the next run instead
of breaking the pipeline permanently.

In [ ]:
bronze_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{checkpoint_path}/_schema")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .load(source_path))

query = (bronze_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(target_table))

query.awaitTermination()
display(spark.table(target_table))

## Continuous vs. triggered
- `trigger(availableNow=True)`: process what's there, then stop — cheap, good
  for hourly/daily Jobs.
- `trigger(processingTime="1 minute")`: keep the stream running and check for
  new files every minute — for near-real-time needs, at the cost of a
  long-running cluster.

In [ ]:
%sql
DESCRIBE HISTORY main.module4.bronze_events_stream

## Monitoring
`query.lastProgress` and `query.recentProgress` expose batch duration, input
rows/sec, and processed rows/sec — useful for spotting a stream that's falling
behind its source.

In [ ]:
print(query.lastProgress)